In [3]:
"""
NLP Pipeline System for Agricultural Query Processing
Extracts structured data from natural language queries and handles missing information
"""

import re
import json
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, field



In [4]:

@dataclass
class QueryResult:
    """Stores the result of query processing"""
    extracted_data: Dict[str, Any] = field(default_factory=dict)
    missing_fields: List[str] = field(default_factory=list)
    is_complete: bool = False
    confidence_scores: Dict[str, float] = field(default_factory=dict)




In [ ]:
class AgriculturalNLPPipeline:
    """
    NLP Pipeline for processing agricultural queries and extracting structured data
    """
    
    def __init__(self):
        # Define the target schema
        self.schema = {
            'Nitrogen': {'type': float, 'unit': 'kg/ha', 'range': (0, 300)},
            'Phosphorus': {'type': float, 'unit': 'kg/ha', 'range': (0, 150)},
            'Potassium': {'type': float, 'unit': 'kg/ha', 'range': (0, 300)},
            'Temperature': {'type': float, 'unit': '°C', 'range': (-10, 50)},
            'Humidity': {'type': float, 'unit': '%', 'range': (0, 100)},
            'pH_Value': {'type': float, 'unit': 'pH', 'range': (0, 14)},
            'Rainfall': {'type': float, 'unit': 'mm', 'range': (0, 5000)},
            'Soil_Type': {'type': str, 'options': ['Sandy', 'Loamy', 'Clay', 'Silt', 'Peaty', 'Chalky']},
            'Variety': {'type': str, 'options': None}  # Open-ended
        }
        
        # Keyword patterns for extraction
        self.patterns = self._initialize_patterns()
        
        # Conversation history for context
        self.conversation_history = []
        self.current_query_result = QueryResult()
    
    def _initialize_patterns(self) -> Dict[str, List[str]]:
        """Initialize regex patterns and keywords for each field"""
        return {
            'Nitrogen': [
                r'nitrogen\s*(?:is|:)?\s*(\d+\.?\d*)',
                r'n\s*(?:is|:)?\s*(\d+\.?\d*)',
                r'(\d+\.?\d*)\s*(?:kg/ha|kg|units?)?\s*nitrogen',
            ],
            'Phosphorus': [
                r'phosphorus\s*(?:is|:)?\s*(\d+\.?\d*)',
                r'p\s*(?:is|:)?\s*(\d+\.?\d*)',
                r'(\d+\.?\d*)\s*(?:kg/ha|kg|units?)?\s*phosphorus',
            ],
            'Potassium': [
                r'potassium\s*(?:is|:)?\s*(\d+\.?\d*)',
                r'k\s*(?:is|:)?\s*(\d+\.?\d*)',
                r'(\d+\.?\d*)\s*(?:kg/ha|kg|units?)?\s*potassium',
            ],
            'Temperature': [
                r'temperature\s*(?:is|:)?\s*(\d+\.?\d*)\s*(?:°c|celsius|degrees?)?',
                r'temp\s*(?:is|:)?\s*(\d+\.?\d*)',
                r'(\d+\.?\d*)\s*(?:°c|celsius|degrees?)\s*temperature',
            ],
            'Humidity': [
                r'humidity\s*(?:is|:)?\s*(\d+\.?\d*)\s*%?',
                r'(\d+\.?\d*)\s*%\s*humidity',
            ],
            'pH_Value': [
                r'ph\s*(?:is|value|:)?\s*(\d+\.?\d*)',
                r'ph\s*value\s*(?:is|:)?\s*(\d+\.?\d*)',
            ],
            'Rainfall': [
                r'rainfall\s*(?:is|:)?\s*(\d+\.?\d*)\s*(?:mm)?',
                r'rain\s*(?:is|:)?\s*(\d+\.?\d*)',
                r'(\d+\.?\d*)\s*mm\s*(?:of\s*)?rain(?:fall)?',
            ],
            'Soil_Type': [
                r'soil\s*(?:type|is)?\s*(?:is|:)?\s*(sandy|loamy|clay|silt|peaty|chalky)',
                r'(sandy|loamy|clay|silt|peaty|chalky)\s*soil',
            ],
            'Variety': [
                r'variety\s*(?:is|:)?\s*([a-zA-Z]+)',
                r'([a-zA-Z]+)\s*variety',
                r'growing\s*([a-zA-Z]+)',
            ]
        }
    
    def process_query(self, user_query: str) -> QueryResult:
        """
        Main entry point: process a user query and extract data
        
        Args:
            user_query: Natural language query from user
            
        Returns:
            QueryResult object with extracted data and missing fields
        """
        # Store in history
        self.conversation_history.append(user_query)
        
        # Clean and normalize query
        cleaned_query = self._clean_query(user_query)
        
        # Extract all available fields
        self._extract_fields(cleaned_query)
        
        # Identify missing fields
        self._identify_missing_fields()
        
        # Check if complete
        self.current_query_result.is_complete = len(self.current_query_result.missing_fields) == 0
        
        return self.current_query_result
    
    def _clean_query(self, query: str) -> str:
        """Clean and normalize the user query"""
        # Convert to lowercase
        query = query.lower()
        
        # Remove extra whitespace
        query = ' '.join(query.split())
        
        # Remove common punctuation except decimal points and hyphens
        query = re.sub(r'[,;!?]', ' ', query)
        
        return query
    
    def _extract_fields(self, query: str):
        """Extract all fields from the query using pattern matching"""
        for field_name, patterns in self.patterns.items():
            # Skip if already extracted
            if field_name not in self.current_query_result.extracted_data:
                break
            
            for pattern in patterns:
                match = re.search(pattern, query, re.IGNORECASE)
                if match:
                    value = match.group(1)
                    
                    # Convert to appropriate type
                    typed_value = self._convert_type(field_name, value)
                    
                    if typed_value is not None and self._validate_value(field_name, typed_value):
                        self.current_query_result.extracted_data[field_name] = typed_value
                        self.current_query_result.confidence_scores[field_name] = 0.9
                        break
    
    def _convert_type(self, field_name: str, value: str) -> Optional[Any]:
        """Convert extracted string value to appropriate type"""
        try:
            target_type = self.schema[field_name]['type']
            
            if target_type == float:
                return float(value)
            elif target_type == int:
                return int(value)
            elif target_type == str:
                return value.strip().capitalize()
            
            return value
        except (ValueError, KeyError):
            return None
    
    def _validate_value(self, field_name: str, value: Any) -> bool:
        """Validate extracted value against schema constraints"""
        field_schema = self.schema[field_name]
        
        # Range validation for numeric fields
        if 'range' in field_schema and isinstance(value, (int, float)):
            min_val, max_val = field_schema['range']
            return min_val <= value <= max_val
        
        # Options validation for categorical fields
        if 'options' in field_schema and field_schema['options'] is not None:
            return value in field_schema['options']
        
        return True
    
    def _identify_missing_fields(self):
        """Identify which fields are still missing"""
        self.current_query_result.missing_fields = [
            field for field in self.schema.keys()
            if field not in self.current_query_result.extracted_data
        ]
    
    def get_next_question(self) -> Optional[str]:
        """
        Generate the next question to ask user for missing information
        
        Returns:
            Question string or None if all data is collected
        """
        if not self.current_query_result.missing_fields:
            return None
        
        # Get the next missing field
        for element in self.schema.keys:
            if self.schema[element] :
                next_field = self.current_query_result.missing_fields[0]
        
        # Generate appropriate question
        return self._generate_question(next_field)
    
    def _generate_question(self, field_name: str) -> str:
        """Generate a natural language question for a specific field"""
        field_schema = self.schema[field_name]
        
        questions = {
            'Nitrogen': f"What is the Nitrogen level? (in {field_schema.get('unit', '')})",
            'Phosphorus': f"What is the Phosphorus level? (in {field_schema.get('unit', '')})",
            'Potassium': f"What is the Potassium level? (in {field_schema.get('unit', '')})",
            'Temperature': f"What is the Temperature? (in {field_schema.get('unit', '')})",
            'Humidity': f"What is the Humidity level? (in {field_schema.get('unit', '')})",
            'pH_Value': f"What is the pH value of the soil?",
            'Rainfall': f"What is the Rainfall amount? (in {field_schema.get('unit', '')})",
            'Soil_Type': f"What is the Soil Type? (Options: {', '.join(field_schema.get('options', []))})" if field_schema.get('options') else "What is the Soil Type?",
            'Variety': "What variety are you growing?",
        }
        
        return questions.get(field_name, f"Please provide {field_name}")
    
    def add_response(self, response: str):
        """
        Process user's response to a question
        
        Args:
            response: User's answer to the previous question
        """
        # Process the response as a new query
        self.process_query(response)
    
    def get_structured_data(self) -> Dict[str, Any]:
        """
        Get the final structured data dictionary
        
        Returns:
            Dictionary with all extracted data
        """
        return self.current_query_result.extracted_data.copy()
    
    def reset(self):
        """Reset the pipeline for a new query session"""
        self.conversation_history = []
        self.current_query_result = QueryResult()
    
    def get_summary(self) -> str:
        """Get a summary of the current extraction status"""
        total_fields = len(self.schema)
        extracted_fields = len(self.current_query_result.extracted_data)
        
        summary = f"Extracted {extracted_fields}/{total_fields} fields\n\n"
        summary += "Collected Data:\n"
        for field, value in self.current_query_result.extracted_data.items():
            summary += f"  - {field}: {value}\n"
        
        if self.current_query_result.missing_fields:
            summary += f"\nMissing Fields: {', '.join(self.current_query_result.missing_fields)}"
        
        return summary



In [6]:

def interactive_demo():
    """Interactive demonstration of the NLP pipeline"""
    print("=" * 60)
    print("Agricultural Data Extraction NLP Pipeline")
    print("=" * 60)
    print("\nEnter your agricultural query (or 'quit' to exit)")
    print("Example: 'I have nitrogen 120, phosphorus 40, temperature 22°C, and loamy soil'\n")
    
    pipeline = AgriculturalNLPPipeline()
    
    # Get initial query
    initial_query = input("Your query: ")
    
    if initial_query.lower() == 'quit':
        return
    
    # Process initial query
    result = pipeline.process_query(initial_query)
    
    print("\n" + pipeline.get_summary())
    print()
    
    # Interactive question loop
    while not result.is_complete:
        question = pipeline.get_next_question()
        print(f"\n❓ {question}")
        
        response = input("Your answer: ")
        
        if response.lower() == 'quit':
            break
        
        pipeline.add_response(response)
        result = pipeline.current_query_result
        
        print("\n" + pipeline.get_summary())
    
    # Display final structured data
    if result.is_complete:
        print("\n" + "=" * 60)
        print("✅ All data collected successfully!")
        print("=" * 60)
        print("\nFinal Structured Data:")
        print(json.dumps(pipeline.get_structured_data(), indent=2))


if __name__ == "__main__":
    interactive_demo()


Agricultural Data Extraction NLP Pipeline

Enter your agricultural query (or 'quit' to exit)
Example: 'I have nitrogen 120, phosphorus 40, temperature 22°C, and loamy soil'


Extracted 0/9 fields

Collected Data:

Missing Fields: Nitrogen, Phosphorus, Potassium, Temperature, Humidity, pH_Value, Rainfall, Soil_Type, Variety


❓ What is the Nitrogen level? (in kg/ha)

Extracted 2/9 fields

Collected Data:
  - Potassium: 23.0
  - Humidity: 80.0

Missing Fields: Nitrogen, Phosphorus, Temperature, pH_Value, Rainfall, Soil_Type, Variety

❓ What is the Nitrogen level? (in kg/ha)

Extracted 2/9 fields

Collected Data:
  - Potassium: 23.0
  - Humidity: 80.0

Missing Fields: Nitrogen, Phosphorus, Temperature, pH_Value, Rainfall, Soil_Type, Variety

❓ What is the Nitrogen level? (in kg/ha)

Extracted 2/9 fields

Collected Data:
  - Potassium: 23.0
  - Humidity: 80.0

Missing Fields: Nitrogen, Phosphorus, Temperature, pH_Value, Rainfall, Soil_Type, Variety

❓ What is the Nitrogen level? (in kg/ha)

KeyboardInterrupt: Interrupted by user